# Reproducible scripts for Operability, Controls, and Dynamics

This companion notebook was generated from fenced `python` code blocks in `chapters/ch21_operability_and_dynamics/chapter.md`. Blocks marked `noexec` in the chapter are preserved for reading but are not executed here.


In [1]:
from pathlib import Path
import re
import shutil

try:
    from IPython.display import Image, display
except Exception:
    Image = None
    display = None

_CHAPTER_FIGURES_DIR = Path("../figures")
_CHAPTER_FIGURES_DIR.mkdir(parents=True, exist_ok=True)
_PAPERLAB_PATTERNS = ("*.png", "*.jpg", "*.jpeg", "*.svg", "*.webp", "*.csv", "*.json", "*.txt", "*.html")
_PAPERLAB_IMAGE_SUFFIXES = {".png", ".jpg", ".jpeg"}


def _paperlab_stamp(path):
    try:
        stat = path.stat()
        return (stat.st_mtime_ns, stat.st_size)
    except OSError:
        return None


def _paperlab_scan_files():
    files = {}
    roots = [Path("."), _CHAPTER_FIGURES_DIR]
    for root in roots:
        if not root.exists():
            continue
        for pattern in _PAPERLAB_PATTERNS:
            for path in root.glob(pattern):
                if path.is_file():
                    files[path.resolve()] = _paperlab_stamp(path)
    return files


def _paperlab_safe_name(name):
    cleaned = re.sub(r"[^A-Za-z0-9_.-]+", "_", name).strip("._")
    return cleaned or "notebook_output"


_paperlab_seen_files = _paperlab_scan_files()


def _paperlab_capture_new_files(label):
    global _paperlab_seen_files
    current = _paperlab_scan_files()
    changed = []
    for path, stamp in current.items():
        if _paperlab_seen_files.get(path) != stamp:
            changed.append(Path(path))
    _paperlab_seen_files = current

    for path in sorted(changed):
        suffix = path.suffix.lower()
        if suffix in _PAPERLAB_IMAGE_SUFFIXES:
            if path.parent.resolve() == _CHAPTER_FIGURES_DIR.resolve():
                dest = path
            else:
                dest = _CHAPTER_FIGURES_DIR / _paperlab_safe_name(f"{label}_{path.name}")
                shutil.copy2(path, dest)
            print(f"Captured figure: figures/{dest.name}")
            if Image is not None and display is not None:
                display(Image(filename=str(dest)))
        else:
            print(f"Generated result file: {path}")

## Example 1 from `chapters/ch21_operability_and_dynamics/chapter.md` line 116


This block is preserved but not executed because it is noexec marker.

````python
from neqsim import jneqsim as J

# Direct Java access through neqsim-python. Use explicit units and call
# setMixingRule before running flashes or process equipment.
# Dynamic ramp study: PI level controller on a hydrogen knockout drum.
fluid = J.thermo.system.SystemSrkEos(298.15, 30.0)
fluid.addComponent("hydrogen", 0.85)
fluid.addComponent("water", 0.15)
fluid.setMixingRule("classic")

feed = J.process.equipment.stream.Stream("KO feed", fluid)
feed.setFlowRate(5000.0, "kg/hr")

drum = J.process.equipment.separator.Separator("V-200 KO drum", feed)
liquid_valve = J.process.equipment.valve.ThrottlingValve("LV-200", drum.getLiquidOutStream())
liquid_valve.setOutletPressure(5.0)

# Attach a PI controller measuring drum liquid level, manipulating valve opening.
lt = J.process.measurementdevice.LevelTransmitter("LT-200", drum)
pi = J.process.controllerdevice.ControllerDeviceBaseClass()
pi.setControllerSetPoint(0.50)  # 50% level setpoint
pi.setControllerParameters(2.0, 60.0, 0.0)  # Kp, Ti (s), Td
pi.setReverseActing(False)
pi.setTransmitter(lt)
liquid_valve.addController("LC-200", pi)

process = J.process.processmodel.ProcessSystem()
for unit in [feed, drum, liquid_valve]:
    process.add(unit)
process.run()

# Run a 600 s dynamic transient with 10 s steps and a feed-rate step disturbance.
process.setTimeStep(10.0)
for step in range(60):
    if step == 12:
        feed.setFlowRate(7500.0, "kg/hr")  # +50% feed step at t=120s
    process.runTransient()
    if step % 10 == 0:
        print("t (s)", (step + 1) * 10,
              "level (-)", lt.getMeasuredValue(),
              "valve opening (%)", liquid_valve.getPercentValveOpening())

````
